<a href="https://colab.research.google.com/github/ammar-aa/Fly_rank_internship_repo/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
from google.colab import userdata
auth=userdata.get("HF_TOKEN")
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
con=duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{auth}'
);
""")
from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
df = con.sql(f"""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
dfF = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
dfM = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()

In [5]:
df['ctr'] = df['gsc_clicks'] / df['gsc_impressions']
df['avg_engagement_sec_per_session'] = df['ga4_total_engagement_sec'] / df['ga4_sessions']
df['scroll_rate'] = df['scroll_events'] / df['ga4_pageviews']

for col in ['ctr', 'avg_engagement_sec_per_session', 'scroll_rate']:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    df[col] = df[col].astype('float64')

print(df['ctr'].dtype)
print(df['avg_engagement_sec_per_session'].dtype)
print(df['scroll_rate'].dtype)

float64
float64
float64


In [6]:
df_trend = dfM.merge(dfF, on=['client_hash_id', 'content_hash_id'], suffixes=('_feb', '_mar'), how='outer')

In [7]:
df_trend = df_trend[df_trend['gsc_impressions_feb'] >= 30]
df_trend = df_trend[df_trend['gsc_impressions_mar'] > 0]

df_trend['trend_pct'] = (
    (df_trend['gsc_impressions_mar'] - df_trend['gsc_impressions_feb'])
    / df_trend['gsc_impressions_feb']
) * 100

clip_value = df_trend['trend_pct'].quantile(0.99)
df_trend['trend_pct'] = df_trend['trend_pct'].clip(lower=-clip_value, upper=clip_value)

In [8]:
df['ga4_data_available'] = df['ga4_data_available'].fillna(False)

conditions = [
    df['gsc_data_available'] & df['ga4_data_available'],
    df['gsc_data_available'] & ~df['ga4_data_available'],
    ~df['gsc_data_available'] & df['ga4_data_available'],
    ~df['gsc_data_available'] & ~df['ga4_data_available']
]

tiers = ['Full', 'GSC-only', 'GA4-only', 'unavailable']

df['tier'] = np.select(conditions, tiers, default=None)

In [9]:
df = df.groupby(['client_hash_id', 'content_hash_id'], as_index=False).agg(
    gsc_sum_position=('gsc_sum_position', 'sum'),
    gsc_avg_position=('gsc_avg_position', 'mean'),
    gsc_impressions=('gsc_impressions', 'sum'),
    gsc_clicks=('gsc_clicks', 'sum'),
    ctr=('ctr', 'mean'),
    ga4_pageviews=('ga4_pageviews', 'sum'),
    ga4_sessions=('ga4_sessions', 'sum'),
    ga4_users=('ga4_users', 'sum'),
    ga4_engaged_sessions=('ga4_engaged_sessions', 'sum'),
    ga4_total_engagement_sec=('ga4_total_engagement_sec', 'sum'),
    avg_engagement_sec_per_session=('avg_engagement_sec_per_session', 'mean'),
    scroll_events=('scroll_events', 'sum'),
    gsc_data_available=('gsc_data_available', 'first'),
    ga4_data_available=('ga4_data_available', 'first'),
    tier=('tier', 'first'),
    scroll_rate=('scroll_rate', 'mean')
)

In [10]:
df = df.merge(df_trend[['client_hash_id', 'content_hash_id', 'trend_pct']], on=['client_hash_id', 'content_hash_id'], how='left')

In [11]:
df_full_tier = df[df['tier'] == 'Full']
conditions = [
    df_full_tier['trend_pct'] < -50,
    (df_full_tier['trend_pct'] >= -50) & (df_full_tier['trend_pct'] < -15),
    (df_full_tier['trend_pct'] >= -15) & (df_full_tier['trend_pct'] <= 15),
    (df_full_tier['trend_pct'] > 15) & (df_full_tier['trend_pct'] <= 50),
    df_full_tier['trend_pct'] > 50
]
ranks = ['5-Sharp decline', '4-Mild decline', '3-Flat', '2-Mild growth', '1-Strong growth']
df_full_tier['trend_dir'] = np.select(conditions, ranks, default=None)

In [12]:
display(df_full_tier.groupby('trend_dir').agg(
    {
        'gsc_sum_position': ['mean', 'count'],
        'gsc_avg_position': ['mean', 'count'],
        'gsc_impressions' : ['mean', 'count'],
        'gsc_clicks' : ['mean', 'count'],
        'ga4_pageviews' : ['mean', 'count'],
        'ga4_sessions' : ['mean', 'count'],
        'ga4_users' : ['mean', 'count'],
        'ga4_engaged_sessions' : ['mean', 'count'],
        'ctr' : ['mean', 'count'],
        'ga4_total_engagement_sec' : ['mean', 'count'],
        'avg_engagement_sec_per_session' : ['mean', 'count'],
        'scroll_events' : ['mean', 'count'],
        'scroll_rate' : ['mean', 'count']
    }
))

gsc_sum_position       gsc_avg_position       gsc_impressions  \
                            mean count             mean count            mean   
trend_dir                                                                       
1-Strong growth     38631.748031   254        13.353879   254     3045.051181   
2-Mild growth       45638.026525   377        12.536609   377     4357.331565   
3-Flat              88217.990816   980        14.127915   980     6512.854082   
4-Mild decline     184232.610397  1789        18.189271  1789     8499.656792   
5-Sharp decline    223852.726792  1702        17.747662  1702    10695.907756   

                      gsc_clicks       ga4_pageviews        ...       ctr  \
                count       mean count          mean count  ...      mean   
trend_dir                                                   ...             
1-Strong growth   254  14.846457   254     55.625984   254  ...  0.005111   
2-Mild growth     377  26.655172   377     83.517241   377  ...  0.006050   
3-Flat            980  32.988776   980     92.705102   980  ...  0.005645   
4-Mild decline   1789  26.999441  1789     92.787032  1789  ...  0.004818   
5-Sharp decline  1702  35.373090  1702     78.090482  1702  ...  0.005353   

                      ga4_total_engagement_sec        \
                count                     mean count   
trend_dir                                              
1-Strong growth   254               179.007874   254   
2-Mild growth     377               339.525199   377   
3-Flat            980               364.755102   980   
4-Mild decline   1789               335.869201  1789   
5-Sharp decline  1702               340.943596  1702   

                avg_engagement_sec_per_session       scroll_events        \
                                          mean count          mean count   
trend_dir                                                                  
1-Strong growth                       5.294051   254      7.488189   254   
2-Mild growth                         8.061824   377       9.29443   377   
3-Flat                                7.260497   980     10.503061   980   
4-Mild decline                        7.073323  1788     11.055897  1789   
5-Sharp decline                       5.838988  1698     12.713866  1702   

                scroll_rate        
                       mean count  
trend_dir                          
1-Strong growth    0.146207   254  
2-Mild growth      0.138781   377  
3-Flat             0.134936   980  
4-Mild decline     0.138958  1789  
5-Sharp decline    0.197115  1702  

[5 rows x 26 columns]

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
def build_tier_frames(df):
    df_full = df[df['tier'] == 'Full'].copy()
    df_gsc_only = df[df['tier'] == 'GSC-only'].copy()
    df_ga4_only = df[df['tier'] == 'GA4-only'].copy()

    df_full['ga4_sessions_pct_rank'] = df_full.groupby('client_hash_id')['ga4_sessions'].transform(
        lambda x: x.rank(pct=True)
    )
    df_full['needs_refresh'] = (
        (df_full['trend_pct'] < -15) &
        (df_full['ga4_sessions_pct_rank'] <= 0.15)
    ).astype(int)

    df_gsc_only['needs_refresh'] = (
        (df_gsc_only['trend_pct'] < -15) &
        (df_gsc_only['gsc_avg_position'] > 10)
    ).astype(int)

    df_ga4_only['needs_refresh'] = (
        (df_ga4_only['ga4_sessions'] <= 2) &
        (df_ga4_only['ga4_engaged_sessions'] <= 2)
    ).astype(int)

    return df_full, df_gsc_only, df_ga4_only

In [14]:
df_full, df_gsc_only, df_ga4_only = build_tier_frames(df)

In [15]:
display(df_full['needs_refresh'].value_counts())
display(df_gsc_only['needs_refresh'].value_counts())
display(df_ga4_only['needs_refresh'].value_counts())

,count
needs_refresh,
0,4646
1,529


,count
needs_refresh,
0,72112
1,27196


,count
needs_refresh,
1,620
0,387


In [16]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

df_full_eligible = df_full[df_full['needs_refresh'] == 1].copy()

norm_avg_position = normalize(df_full_eligible['gsc_avg_position'])
norm_ctr = normalize(df_full_eligible['ctr'])
norm_clicks = normalize(df_full_eligible['gsc_clicks'])
norm_engaged_sessions = normalize(df_full_eligible['ga4_engaged_sessions'])
norm_scroll = normalize(df_full_eligible['scroll_rate'])
norm_engagement_sec = normalize(df_full_eligible['avg_engagement_sec_per_session'])
norm_pv_users_ratio = normalize(df_full_eligible['ga4_pageviews'] / df_full_eligible['ga4_users'])

df_full_eligible['score'] = (
     0.25 * norm_avg_position +
    -0.15 * norm_ctr +
     0.10 * norm_clicks +
    -0.117 * norm_engaged_sessions +
    -0.117 * norm_scroll +
    -0.117 * norm_engagement_sec +
    -0.15 * norm_pv_users_ratio
)
display(df_full_eligible[['gsc_avg_position', 'ctr', 'gsc_clicks', 'score']].sort_values('score', ascending=False).head(10))

,gsc_avg_position,ctr,gsc_clicks,score
299588,74.991589,0.000000,0,0.25
249752,68.087775,0.000000,0,0.226853
250725,64.563373,0.000000,0,0.215037
249823,60.124395,0.000000,0,0.200154
298061,53.498227,0.000000,0,0.177939
70241,43.879101,0.000349,7,0.175941
56802,21.563190,0.000868,20,0.164403
303618,49.006836,0.000000,0,0.16288
63220,27.849957,0.003436,16,0.16118
250180,47.527083,0.000000,0,0.157919


In [17]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())


df_gsc_only_eligible = df_gsc_only[df_gsc_only['needs_refresh'] == 1].copy()

norm_sum_position = normalize(df_gsc_only_eligible['gsc_sum_position'])
norm_ctr = normalize(df_gsc_only_eligible['ctr'])
norm_clicks = normalize(df_gsc_only_eligible['gsc_clicks'])

df_gsc_only_eligible['score'] = (
     0.30 * norm_sum_position +
    -0.50 * norm_ctr +
     0.20 * norm_clicks
)
display(df_gsc_only_eligible[['gsc_sum_position', 'ctr', 'gsc_clicks', 'score']].sort_values('score', ascending=False).head(10))


,gsc_sum_position,ctr,gsc_clicks,score
48035,2225082,0.005389,475,0.360921
59650,3306560,0.001371,197,0.345105
68947,3217192,0.001188,163,0.324209
62726,3716426,0.000116,9,0.303387
61003,3593090,0.000011,2,0.290848
47477,3186573,0.000449,60,0.280934
66008,3376049,0.000073,5,0.274375
47625,1939886,0.003288,257,0.253393
64808,2953021,0.000085,5,0.240184
67403,2952777,0.000014,1,0.238726


In [18]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

df_ga4_only_eligible = df_ga4_only[df_ga4_only['needs_refresh'] == 1].copy()

norm_engaged_sessions = normalize(df_ga4_only_eligible['ga4_engaged_sessions'])
norm_users = normalize(df_ga4_only_eligible['ga4_users'])
norm_engagement_sec = normalize(df_ga4_only_eligible['avg_engagement_sec_per_session'])
norm_pageviews = normalize(df_ga4_only_eligible['ga4_pageviews'])
norm_scroll = normalize(df_ga4_only_eligible['scroll_rate'])

df_ga4_only_eligible['score'] = (
    -0.40 * norm_engaged_sessions +
    -0.20 * norm_users +
    -0.15 * norm_engagement_sec +
    -0.15 * norm_pageviews +
    -0.10 * norm_scroll
)
display(df_ga4_only_eligible[['ga4_engaged_sessions', 'ga4_users', 'ga4_pageviews', 'score']].sort_values('score', ascending=False).head(10))

,ga4_engaged_sessions,ga4_users,ga4_pageviews,score
331209,0,1,1,-0.03
331197,0,1,1,-0.03
331013,0,1,1,-0.03
330496,0,1,1,-0.03
302294,0,1,1,-0.03
300019,0,1,1,-0.03
296995,0,1,1,-0.03
296851,0,1,1,-0.03
296356,0,1,1,-0.03
56004,0,1,1,-0.03


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [44]:
def expected_ctr(position):
    if position <= 1: return 0.30
    elif position <= 2: return 0.17
    elif position <= 3: return 0.10
    elif position <= 5: return 0.06
    elif position <= 10: return 0.02
    else: return 0.01


df_full_eligible['expected_ctr'] = df_full_eligible['gsc_avg_position'].apply(expected_ctr)
df_full_eligible['low_ctr_for_position'] = df_full_eligible['ctr'] < (0.5 * df_full_eligible['expected_ctr'])
df_full_eligible['low_position'] = df_full_eligible['gsc_avg_position'] > 10
df_full_eligible['low_engagement'] = (df_full_eligible['ga4_engaged_sessions'] / df_full_eligible['ga4_sessions']) < 0.50
df_full_eligible['low_scroll'] = df_full_eligible['scroll_rate'] == 0
df_full_eligible['low_engagement_time'] = df_full_eligible['avg_engagement_sec_per_session'] == 0

def build_action_text_full(row):
    lines = []
    if row['low_position']:
        lines.append("Ranks below page 1 (position > 10).")
    if row['low_ctr_for_position']:
        lines.append(f"CTR is under half of what's typical for its position (expected ~{row['expected_ctr']:.0%}).")
    if row['low_engagement']:
        lines.append("Engagement rate below 50%.")
    if row['low_scroll']:
        lines.append("Zero recorded clicks despite adequate impressions — CTR/snippet problem.")
    if row['low_engagement_time']:
        lines.append("Average time-per-session in the bottom 25% of flagged pages (relative to this pool).")
    return " ".join(lines) if lines else "No specific underperforming signal identified."

df_full_eligible['action'] = df_full_eligible.apply(build_action_text_full, axis=1)


def build_confidence_note_full(row):
    if row['ga4_sessions'] <= 2:
        return "Low confidence — very few GA4 sessions this month, engagement signals (engagement rate, scroll rate, time-per-session) may be unstable."
    if row['gsc_impressions'] < 100:
        return "Moderate confidence — low search impression volume, trend percentage may be noisy."
    return "High confidence — based on adequate impression and session volume across the tracked period."

def build_wrong_note_full(row):
    if row['trend_pct'] < -80:
        return "Could be wrong if this is a 'remontada' case — a sharp recent decline that may already be recovering on the engagement side, which this snapshot wouldn't fully capture."
    if row['ga4_sessions'] <= 2:
        return "Could be wrong if the low GA4 activity reflects a tracking gap rather than genuine underperformance."
    return "Could be wrong if external factors (seasonality, intentional deprioritization by the client) explain the decline rather than a content quality issue."

df_full_eligible['confidence_note'] = df_full_eligible.apply(build_confidence_note_full, axis=1)
df_full_eligible['what_would_make_it_wrong'] = df_full_eligible.apply(build_wrong_note_full, axis=1)


display(df_full_eligible.head(20))


,client_hash_id,content_hash_id,gsc_sum_position,gsc_avg_position,gsc_impressions,gsc_clicks,ctr,ga4_pageviews,ga4_sessions,ga4_users,...,score,low_position,low_engagement,action,expected_ctr,low_ctr_for_position,low_scroll,low_engagement_time,confidence_note,what_would_make_it_wrong
29165,client_0fa64a184f18a4a0,content_0c4314a6f873408d,2524,5.575615,867,2,0.000208,1,1,1,...,0.026383,False,True,CTR is under half of what's typical for its po...,0.02,True,True,True,Low confidence — very few GA4 sessions this mo...,Could be wrong if the low GA4 activity reflect...
29282,client_0fa64a184f18a4a0,content_1de7f3c5c15b2291,868,3.449133,290,2,0.003415,1,1,1,...,0.012954,False,True,CTR is under half of what's typical for its po...,0.06,True,True,True,Low confidence — very few GA4 sessions this mo...,Could be wrong if this is a 'remontada' case —...
29570,client_0fa64a184f18a4a0,content_497c364b576bfaf5,1733,3.736250,536,1,0.000672,1,1,1,...,0.014543,False,True,CTR is under half of what's typical for its po...,0.06,True,True,True,Low confidence — very few GA4 sessions this mo...,Could be wrong if the low GA4 activity reflect...
30159,client_0fa64a184f18a4a0,content_99ffe79e393286f5,502,2.642652,234,1,0.000866,1,1,1,...,0.010496,False,True,CTR is under half of what's typical for its po...,0.10,True,True,True,Low confidence — very few GA4 sessions this mo...,Could be wrong if the low GA4 activity reflect...
30687,client_0fa64a184f18a4a0,content_ebd699088ccf7224,902,3.163919,347,1,0.000452,1,1,1,...,0.013057,False,True,CTR is under half of what's typical for its po...,0.06,True,True,True,Low confidence — very few GA4 sessions this mo...,Could be wrong if the low GA4 activity reflect...
45088,client_20259bd6705d81d4,content_02358f598df20fe7,58751,25.210395,2157,2,0.001274,2,2,2,...,0.031619,True,True,Ranks below page 1 (position > 10). CTR is und...,0.01,True,False,True,Low confidence — very few GA4 sessions this mo...,Could be wrong if the low GA4 activity reflect...
45097,client_20259bd6705d81d4,content_02895d816b0e3a95,6106,12.561410,868,1,0.004608,1,1,1,...,-0.080601,True,True,Ranks below page 1 (position > 10). CTR is und...,0.01,True,False,True,Low confidence — very few GA4 sessions this mo...,Could be wrong if this is a 'remontada' case —...
45284,client_20259bd6705d81d4,content_0d4211f931b596df,168829,20.183166,6809,1,0.000461,4,3,3,...,0.031099,True,True,Ranks below page 1 (position > 10). CTR is und...,0.01,True,False,True,High confidence — based on adequate impression...,Could be wrong if external factors (seasonalit...
45388,client_20259bd6705d81d4,content_127f40ef6b42cec2,1599,2.542455,1366,0,0.000000,1,1,1,...,-0.109902,False,True,CTR is under half of what's typical for its po...,0.10,True,False,True,Low confidence — very few GA4 sessions this mo...,Could be wrong if this is a 'remontada' case —...
45450,client_20259bd6705d81d4,content_1649abe88c929213,2667,7.316646,523,2,0.004048,4,3,3,...,-0.004573,False,True,CTR is under half of what's typical for its po...,0.02,True,False,True,High confidence — based on adequate impression...,Could be wrong if external factors (seasonalit...


In [45]:

df_gsc_only_eligible['expected_ctr'] = df_gsc_only_eligible['gsc_sum_position'].apply(lambda x: expected_ctr(x))
low_impressions_threshold = df_gsc_only_eligible['gsc_impressions'].quantile(0.25)
df_gsc_only_eligible['low_impressions'] = df_gsc_only_eligible['gsc_impressions'] < low_impressions_threshold
df_gsc_only_eligible['low_clicks'] = df_gsc_only_eligible['gsc_clicks'] == 0

def build_action_text_gsc(row):
    lines = []
    if row['low_impressions']:
        lines.append("Search impressions in the bottom 25% of flagged pages — visibility problem.")
    if row['low_clicks'] and not row['low_impressions']:
        lines.append("Zero recorded clicks despite adequate impressions — CTR/snippet problem.")
    return " ".join(lines) if lines else "No specific underperforming signal identified."

df_gsc_only_eligible['action'] = df_gsc_only_eligible.apply(build_action_text_gsc, axis=1)


def build_confidence_note_gsc(row):
    if row['gsc_impressions'] < 100:
        return "Low confidence — very low search impression volume, trend percentage and CTR may be noisy."
    if row['gsc_clicks'] == 0:
        return "Moderate confidence — zero clicks recorded, so CTR-based diagnosis has limited signal."
    return "High confidence — based on adequate impression and click volume across the tracked period."

def build_wrong_note_gsc(row):
    if row['trend_pct'] < -80:
        return "Could be wrong if this is a 'remontada' case — a sharp recent decline that may already be recovering, which this snapshot wouldn't capture."
    if row['gsc_impressions'] < 100:
        return "Could be wrong if the low impression count reflects a small/niche keyword rather than a real visibility failure."
    return "Could be wrong if external factors (seasonality, SERP feature changes, intentional deprioritization) explain the decline rather than a content quality issue."

df_gsc_only_eligible['confidence_note'] = df_gsc_only_eligible.apply(build_confidence_note_gsc, axis=1)
df_gsc_only_eligible['what_would_make_it_wrong'] = df_gsc_only_eligible.apply(build_wrong_note_gsc, axis=1)


display(df_gsc_only_eligible.head(20))

,client_hash_id,content_hash_id,gsc_sum_position,gsc_avg_position,gsc_impressions,gsc_clicks,ctr,ga4_pageviews,ga4_sessions,ga4_users,...,trend_pct,needs_refresh,score,low_impressions,low_ctr_despite_impressions,action,expected_ctr,low_clicks,confidence_note,what_would_make_it_wrong
7,client_0797ff3a1fc9a6a5,content_04c67f3541177192,4759,14.129210,331,2,0.008193,0,0,0,...,-25.679758,1,-0.027200,False,False,No specific underperforming signal identified.,0.01,False,High confidence — based on adequate impression...,Could be wrong if external factors (seasonalit...
18,client_0797ff3a1fc9a6a5,content_1207efddce873942,6679,14.859827,461,0,0.000000,0,0,0,...,-62.255965,1,0.000525,False,False,Zero recorded clicks despite adequate impressi...,0.01,True,"Moderate confidence — zero clicks recorded, so...",Could be wrong if external factors (seasonalit...
22,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,2775,12.297523,232,0,0.000000,0,0,0,...,-29.310345,1,0.000210,False,False,Zero recorded clicks despite adequate impressi...,0.01,True,"Moderate confidence — zero clicks recorded, so...",Could be wrong if external factors (seasonalit...
178,client_0797ff3a1fc9a6a5,content_be06033d30b49299,111186,51.733909,2092,1,0.000520,0,0,0,...,-26.051625,1,0.007578,False,False,No specific underperforming signal identified.,0.01,False,High confidence — based on adequate impression...,Could be wrong if external factors (seasonalit...
252,client_0797ff3a1fc9a6a5,content_fa84e03e3d5e2fa2,19567,11.327251,1935,7,0.006331,0,0,0,...,-33.281654,1,-0.017443,False,False,No specific underperforming signal identified.,0.01,False,High confidence — based on adequate impression...,Could be wrong if external factors (seasonalit...
293,client_08a6a72ff48e62c0,content_004cb3be302799bb,2672,41.259570,68,0,0.000000,0,0,0,...,-30.882353,1,0.000202,True,False,Search impressions in the bottom 25% of flagge...,0.01,True,Low confidence — very low search impression vo...,Could be wrong if the low impression count ref...
294,client_08a6a72ff48e62c0,content_004cba86860ab130,11517,30.000587,421,2,0.003072,0,0,0,...,-28.741093,1,-0.008897,False,False,No specific underperforming signal identified.,0.01,False,High confidence — based on adequate impression...,Could be wrong if external factors (seasonalit...
297,client_08a6a72ff48e62c0,content_0050d70972c911bf,19456,51.283316,370,0,0.000000,0,0,0,...,-23.783784,1,0.001556,False,False,Zero recorded clicks despite adequate impressi...,0.01,True,"Moderate confidence — zero clicks recorded, so...",Could be wrong if external factors (seasonalit...
302,client_08a6a72ff48e62c0,content_005de32a6050545f,56569,35.667429,1604,2,0.001126,0,0,0,...,-29.488778,1,0.001488,False,False,No specific underperforming signal identified.,0.01,False,High confidence — based on adequate impression...,Could be wrong if external factors (seasonalit...
304,client_08a6a72ff48e62c0,content_00647b17c6a4f894,7006,46.326359,142,0,0.000000,0,0,0,...,-98.591549,1,0.000551,True,False,Search impressions in the bottom 25% of flagge...,0.01,True,"Moderate confidence — zero clicks recorded, so...",Could be wrong if this is a 'remontada' case —...


In [46]:
df_ga4_only_eligible['action'] = (
    "Insufficient GA4 traffic (median session count is very low in this tier) to identify a "
    "specific underperforming signal — recommend building visibility/promotion before a content-level refresh."
)

display(df_ga4_only_eligible.head(20))


def build_confidence_note_ga4(row):
    return "Low confidence — this tier's pages have no GSC data and typically very low GA4 traffic (median ~2 sessions), so this flag reflects data scarcity as much as genuine underperformance."

def build_wrong_note_ga4(row):
    return "Could be wrong if this page simply has insufficient tracking history/traffic to judge — more data may reveal it is performing adequately, or the low activity reflects the page's niche rather than a real problem."

df_ga4_only_eligible['confidence_note'] = df_ga4_only_eligible.apply(build_confidence_note_ga4, axis=1)
df_ga4_only_eligible['what_would_make_it_wrong'] = df_ga4_only_eligible.apply(build_wrong_note_ga4, axis=1)

,client_hash_id,content_hash_id,gsc_sum_position,gsc_avg_position,gsc_impressions,gsc_clicks,ctr,ga4_pageviews,ga4_sessions,ga4_users,...,tier,scroll_rate,trend_pct,needs_refresh,score,low_traffic,low_engagement_despite_traffic,action,low_reach,low_depth
28646,client_08d2847f24cf89c1,content_41aae810179c33d4,0,NaN,0,0,NaN,2,2,2,...,GA4-only,0.500000,NaN,1,-0.310293,False,True,Insufficient GA4 traffic (median session count...,False,False
28808,client_08d2847f24cf89c1,content_a920c689fe8380dd,0,NaN,0,0,NaN,2,2,2,...,GA4-only,0.500000,NaN,1,-0.31,False,True,Insufficient GA4 traffic (median session count...,False,False
28981,client_0e1acc6cd57b0eba,content_5eb4a469a0571df7,0,NaN,0,0,NaN,2,2,2,...,GA4-only,0.000000,NaN,1,-0.26,False,True,Insufficient GA4 traffic (median session count...,False,False
29033,client_0e1acc6cd57b0eba,content_b033547d1ed050c4,0,NaN,0,0,NaN,2,2,2,...,GA4-only,0.000000,NaN,1,-0.26,False,True,Insufficient GA4 traffic (median session count...,False,False
29078,client_0e1acc6cd57b0eba,content_f8cd6b15cd8bab8e,0,NaN,0,0,NaN,2,2,2,...,GA4-only,0.500000,NaN,1,-0.310117,False,True,Insufficient GA4 traffic (median session count...,False,False
30557,client_0fa64a184f18a4a0,content_d9a4d4e5d92ea890,219,5.612281,38,1,0.017544,1,1,1,...,GA4-only,0.000000,NaN,1,-0.03,False,True,Insufficient GA4 traffic (median session count...,False,False
45746,client_20259bd6705d81d4,content_278dbac90c492316,1536,9.351044,356,0,0.000000,1,1,1,...,GA4-only,1.000000,-98.314607,1,-0.13,False,True,Insufficient GA4 traffic (median session count...,False,False
45830,client_20259bd6705d81d4,content_2c55628e48289043,1234,8.283943,237,0,0.000000,3,2,2,...,GA4-only,0.666667,-90.717300,1,-0.356667,False,True,Insufficient GA4 traffic (median session count...,False,False
46504,client_20259bd6705d81d4,content_500a572d5d9ca168,813,8.717769,145,0,0.000000,2,2,2,...,GA4-only,0.500000,-97.931034,1,-0.31,False,True,Insufficient GA4 traffic (median session count...,False,False
46962,client_20259bd6705d81d4,content_6764183d7de96136,2903,38.582412,83,0,0.000000,2,2,2,...,GA4-only,1.000000,NaN,1,-0.36,False,True,Insufficient GA4 traffic (median session count...,False,False


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The row-1 GSC-only case (client_e547b89c05043229 / content thread, gsc_clicks=475, ctr=0.54%, trend_pct=-89.1%) initially looked like a weak pick — decent CTR and click volume suggested a healthy page. Investigation showed the page likely had much higher traffic before an 89% collapse, so residual clicks look large only in comparison to smaller pages. Also weak by design: any GA4-only tier pick (median 2 sessions, 0 engaged sessions) — flagged more by data scarcity than confirmed underperformance, as already disclosed.

In [47]:
excluded_cols = ['is_declining_label', 'trend_direction', 'needs_refresh_flag']  # any known product/label flags

for name, df_check in [('df_full_eligible', df_full_eligible),
                         ('df_gsc_only_eligible', df_gsc_only_eligible),
                         ('df_ga4_only_eligible', df_ga4_only_eligible)]:
    leaked = [c for c in excluded_cols if c in df_check.columns]
    print(f"{name}: {'LEAK FOUND -> ' + str(leaked) if leaked else 'clean'}")

print("trend_pct built from: month=2026-02 vs month=2026-03 only (per df_trend construction) — confirmed by code inspection, no fact_content_query_90d or June data referenced anywhere in this pipeline.")

df_full_eligible: clean
df_gsc_only_eligible: clean
df_ga4_only_eligible: clean
trend_pct built from: month=2026-02 vs month=2026-03 only (per df_trend construction) — confirmed by code inspection, no fact_content_query_90d or June data referenced anywhere in this pipeline.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.